# Импорт модулей

In [67]:
import os
import pickle
import pandas as pd
import numpy as np
import time
from datetime import datetime
from tqdm import tqdm

from config import PERIODS, METHODS, TRADING_DAYS, SHARPE_THRESHOLD, SORTINO_THRESHOLD
from utils import (
    load_prices_dict, get_dividends_sum,
    annual_return_from_series, compute_sharpe_with_dividends, sortino_with_dividends,
    ensure_dirs
)

ensure_dirs(['data', 'results/tables', 'results/figures'])

# Загрузка словарей цен и выявление общих тикеров

In [69]:
dict_2023_2024 = load_prices_dict(os.path.join('data', 'prices_dict_2023_2024.pkl'))
dict_2024_2025 = load_prices_dict(os.path.join('data', 'prices_dict_2024_2025.pkl'))

tickers_2023 = set(dict_2023_2024.keys())
tickers_2024 = set(dict_2024_2025.keys())

common_tickers = tickers_2023 & tickers_2024
print(f"Общих тикеров: {len(common_tickers)}")
print(f"Только в 2023-2024: {len(tickers_2023 - tickers_2024)}")
print(f"Только в 2024-2025: {len(tickers_2024 - tickers_2023)}")

Общих тикеров: 242
Только в 2023-2024: 0
Только в 2024-2025: 13


# Расчёт годовой доходности для общих тикеров (с дивидендами) ~ 4 минуты

In [71]:
results = []
for ticker in tqdm(list(common_tickers), desc="Обработка общих тикеров"):
    for period_name, period in PERIODS.items():
        prices_series = dict_2023_2024.get(ticker) if period_name == '2023_2024' else dict_2024_2025.get(ticker)
        if prices_series is None or len(prices_series) < 5:
            continue
        start_date = datetime.strptime(period['start'], '%Y-%m-%d')
        end_date = datetime.strptime(period['end'], '%Y-%m-%d')
        dividends = get_dividends_sum(ticker, start_date, end_date)
        ret = annual_return_from_series(prices_series, dividends)
        if ret is not None:
            results.append({
                'ticker': ticker,
                'period': period_name,
                'annual_return_pct': round(ret, 2),
                'risk_free_rate_pct': period['rf'] * 100,
                'excess': round(ret - period['rf'] * 100, 2),
                'beats_rf': ret > period['rf'] * 100
            })
        time.sleep(0.1)

df_annual = pd.DataFrame(results)
df_annual.to_csv('results/tables/annual_returns_comparison.csv', index=False)
print("Сохранено results/tables/annual_returns_comparison.csv")

Обработка общих тикеров: 100%|███████████████████████████████████████████████████████| 242/242 [01:30<00:00,  2.67it/s]

Сохранено results/tables/annual_returns_comparison.csv


# Фильтрация акций, превысивших безрисковую ставку в оба периода

In [73]:
df_beat = df_annual[df_annual['beats_rf'] == True]
pivot = df_beat.pivot(index='ticker', columns='period', values='excess').dropna()
common_beat = pivot.index.tolist()
print(f"Акций с доходностью > rf в обоих периодах: {len(common_beat)}")
pd.DataFrame({'ticker': common_beat}).to_csv('results/tables/common_beats_both_periods.csv', index=False)

Акций с доходностью > rf в обоих периодах: 31


# Расчёт коэффициента Шарпа (с дивидендами) для этих акций

In [75]:
sharpe_results = []
for ticker in tqdm(common_beat, desc="Проверка Sharpe > 0.75"):
    ok = True
    row = {'ticker': ticker}
    for period_name, period in PERIODS.items():
        prices_series = dict_2023_2024.get(ticker) if period_name == '2023_2024' else dict_2024_2025.get(ticker)
        if prices_series is None:
            ok = False
            break
        start_date = datetime.strptime(period['start'], '%Y-%m-%d')
        end_date = datetime.strptime(period['end'], '%Y-%m-%d')
        dividends = get_dividends_sum(ticker, start_date, end_date)
        sharpe = compute_sharpe_with_dividends(prices_series, dividends, period['rf'])
        if sharpe is None or sharpe <= SHARPE_THRESHOLD:
            ok = False
            break
        row[f'sharpe_{period_name}'] = sharpe
    if ok:
        sharpe_results.append(row)

df_sharpe = pd.DataFrame(sharpe_results)
if not df_sharpe.empty:
    df_sharpe.to_csv('results/tables/common_beats_sharpe_gt_0.75.csv', index=False)
    print(f"Акций с Sharpe > 0.75 в обоих периодах: {len(df_sharpe)}")
else:
    print("Нет акций, удовлетворяющих условию Sharpe > 0.75")

Проверка Sharpe > 0.75: 100%|██████████████████████████████████████████████████████████| 31/31 [00:04<00:00,  6.47it/s]

Акций с Sharpe > 0.75 в обоих периодах: 21


# Расчёт коэффициента Сортино для отфильтрованных акций

In [77]:
tickers_sharpe = pd.read_csv('results/tables/common_beats_sharpe_gt_0.75.csv')['ticker'].tolist() if os.path.exists('results/tables/common_beats_sharpe_gt_0.75.csv') else []
if not tickers_sharpe:
    raise ValueError("Нет тикеров с Sharpe > 0.75. Запустите предыдущие ячейки.")

sortino_results = []
for ticker in tqdm(tickers_sharpe, desc="Расчёт Sortino"):
    row = {'ticker': ticker}
    for period_name, period in PERIODS.items():
        prices_series = dict_2023_2024.get(ticker) if period_name == '2023_2024' else dict_2024_2025.get(ticker)
        if prices_series is None:
            row[f'sortino_{period_name}'] = None
            continue
        start_date = datetime.strptime(period['start'], '%Y-%m-%d')
        end_date = datetime.strptime(period['end'], '%Y-%m-%d')
        dividends = get_dividends_sum(ticker, start_date, end_date)
        sortino = sortino_with_dividends(prices_series, dividends, period['rf'])
        row[f'sortino_{period_name}'] = sortino
    sortino_results.append(row)

df_sortino = pd.DataFrame(sortino_results)
df_sortino['min_sortino'] = df_sortino[['sortino_2023_2024', 'sortino_2024_2025']].min(axis=1)
df_sortino = df_sortino.sort_values('min_sortino', ascending=False)
df_sortino.to_csv('results/tables/common_beats_sortino.csv', index=False)
print(f"Результаты Sortino сохранены, всего записей: {len(df_sortino)}")

Расчёт Sortino: 100%|██████████████████████████████████████████████████████████████████| 21/21 [00:03<00:00,  6.72it/s]

Результаты Sortino сохранены, всего записей: 21


# Отбор топ-21 акции по Sortino

In [79]:
df_sortino = pd.read_csv('results/tables/common_beats_sortino.csv')
top21 = df_sortino.head(21)['ticker'].tolist()
print(f"Отобрано {len(top21)} акций для кластеризации и построения портфелей.")
pd.DataFrame({'ticker': top21}).to_csv('results/tables/top21_tickers.csv', index=False)

Отобрано 21 акций для кластеризации и построения портфелей.


# Формирование датасетов цен (inner join и forward fill) для каждого периода и метода ~3 мин

In [90]:
def create_price_df(tickers, period_name, method):
    pkl_path = os.path.join('data', PERIODS[period_name]['pkl_file'])
    prices_dict = load_prices_dict(pkl_path)
    prices_dict = {t: prices_dict[t] for t in tickers if t in prices_dict}
    df = pd.DataFrame({t: s for t, s in prices_dict.items()})
    if method == 'inner':
        df = df.dropna()
    elif method == 'ffill':
        df = df.ffill().bfill()
    else:
        raise ValueError("method должен быть 'inner' или 'ffill'")
    out_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
    df.to_csv(out_file)
    print(f"Создан {out_file}: {df.shape[1]} акций, {len(df)} дней")
    return df

print("\nПромежуточные итоги")
tickers = top21
for period_name in PERIODS.keys():
    for method in METHODS:
        create_price_df(tickers, period_name, method)


Промежуточные итоги
Создан data\sortino_tickers_prices_2023_2024_inner.csv: 21 акций, 115 дней
Создан data\sortino_tickers_prices_2023_2024_ffill.csv: 21 акций, 116 дней
Создан data\sortino_tickers_prices_2024_2025_inner.csv: 21 акций, 112 дней
Создан data\sortino_tickers_prices_2024_2025_ffill.csv: 21 акций, 115 дней
